# Self-reported expressibility predicts communicative success: Open dataset, validation, and simulation
## Data preparation of guessing responses from referential experiment

Author: [Anonymized for review]<br>
Date: 1. 6. 2025

In [3]:
import os
import glob
import pandas as pd

curfolder = os.getcwd()
rawdata = curfolder + '\\..\\rawdata\\'
dataset = curfolder + '\\..\\dataset\\'

In [ ]:
responsedata = glob.glob(rawdata + '*.csv')

# ignore all csv files with _broken or _reinitiated, plus pilot data (see README for details)
responsedata = [file for file in responsedata if '_broken' not in file and '_reinitiated' not in file and '0_1' not in file and '0_2' not in file]

#print(responsedata[0:10])

In [ ]:
df_all = pd.DataFrame()

pcn_count = 1

for file in responsedata:
    print('working on ' + file) 

    df = pd.read_csv(file)
    # get rid of all rows where in only NA
    df = df.dropna(how='all')
    # make a column order to keep track of order of stimuli presentation
    df['trial_order'] = range(1, len(df) + 1)
    # keep only columns practice, word, modality, answer
    df = df[['trial_order', 'practice', 'cycle', 'word', 'modality', 'answer', 'correction']]
    # get the filename
    filename = file.split('\\')[-1].split('.')[0]

    # rename cycle to participant
    df = df.rename(columns={'cycle': 'participant'})
    # convert values in participant to integers
    df['participant'] = df['participant'].astype(int)

    # rename practice to trial_type
    df = df.rename(columns={'practice': 'trial_type'})
    df['trial_type'] = df['trial_type'].replace('none', 'target')

    # convert correction to integer
    df['correction'] = df['correction'].fillna(0)
    df['correction'] = df['correction'].astype(int)

    # add unique identifier for each session
    sessionID = filename.split('_')[0] + '_' + filename.split('_')[2] 
    sessionID = sessionID.replace('part', '')
    df['sessionID'] = sessionID

    # add unique identifier for each experiment part
    df['exp_part'] = sessionID.split('_')[1]
    # add unique identifier for each dyad
    df['dyad'] = sessionID.split('_')[0]

    # add unique identifier for each participant
    df['pcnID'] = df['dyad'] + '_' + df['participant'].astype(str)

    df_all = pd.concat([df_all, df])

# create pcn for each unique pcnID starting from 1
pcn = df_all['pcnID'].unique()
#df_all['pcnID'] = 0
for p in pcn:
    df_all.loc[df_all['pcnID'] == p, 'pcnID'] = pcn_count
    pcn_count += 1

# reset index
df_all = df_all.reset_index(drop=True)

df_all.to_csv(dataset + '\\all_data_raw.csv', index=False)
df_all.head(15)
    
